# Lesson 11 · AI Agents II · MCP · Demo
## As ferramentas da aula 10, agora atrás de um protocolo

**DCA0305 · Machine Learning-Based Systems Design · UFRN**

Na aula 10, as três ferramentas do estoque (buscar, detalhar e comparar) moravam dentro do notebook do agente. Escrevemos à mão a descrição de cada uma em JSON Schema e um dicionário ligando o nome à função. Hoje elas viram um **servidor MCP**, um programa separado que qualquer host consegue usar.

| Parte | O que fazemos | Usa o modelo? |
|---|---|---|
| **1 · O servidor** | as três funções da aula 10 viram um `server.py` com o FastMCP | não |
| **2 · Inspecionar** | listamos e chamamos as ferramentas pela linha de comando | não |
| **3 · JSON-RPC na mão** | conversamos com o servidor escrevendo as mensagens nós mesmos | não |
| **4 · Um cliente MCP** | o mesmo, agora com o cliente Python do FastMCP | não |
| **5 · O agente** | o agente da aula 10, com as ferramentas vindas do servidor | sim, `qwen2.5:3b` no Ollama |

No fim, o pedido da aula 10 (*"Quero um SUV para a família, até uns 80 mil"*) tem a mesma resposta, `c082`, `c081` e `c109`. A diferença é que agora as ferramentas chegam por um protocolo padrão.

**Como rodar.** No Colab, escolha o runtime **T4 GPU** (Ambiente de execução → Alterar o tipo de ambiente de execução) e rode tudo de cima a baixo. Só a Parte 5 usa o modelo.

---
## 🧰 Preparação

A célula abaixo confere cada passo e só faz o que falta. Instala o FastMCP (a biblioteca Python de MCP), baixa o estoque `cars.csv` da aula 10 e prepara o Ollama com o `qwen2.5:3b`, como na aula passada. Pode rodar quantas vezes quiser. Na primeira vez no Colab, leva uns 4 minutos. Se ela pedir para **reiniciar a sessão**, é porque o `pip` atualizou uma biblioteca que já estava carregada: reinicie e rode a célula de novo.

In [1]:
# @title 🧰 Preparação: confere o FastMCP, o estoque, o Ollama e o modelo, e só instala o que faltar
import os, shutil, subprocess, sys, time, requests

FASTMCP_VERSION = "4.1.4"
OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen2.5:3b"
IN_COLAB = "google.colab" in sys.modules
sh = get_ipython().system                      # roda um comando de shell mostrando a saída na célula

os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]  # para o !fastmcp achar o programa

# 1 · O FastMCP está instalado e carrega por inteiro?
RESTART = ("o pip atualizou bibliotecas que esta sessão já tinha carregado. Reinicie a sessão "
           "(no Colab, Ambiente de execução → Reiniciar sessão; no Jupyter, Kernel → Restart) "
           "e rode esta célula de novo.")
try:
    import fastmcp
    print(f"✅ FastMCP {fastmcp.__version__} já instalado")
except ImportError:
    print(f"⏳ instalando o FastMCP {FASTMCP_VERSION}...")
    sh(f"pip -q install --disable-pip-version-check fastmcp=={FASTMCP_VERSION}")
    try:
        import fastmcp
    except Exception as e:
        raise RuntimeError(RESTART) from e
    print(f"✅ FastMCP {fastmcp.__version__} instalado")
try:
    from fastmcp import Client     # o cliente puxa mais bibliotecas (anyio, typing_extensions...); uma versão velha na memória quebra aqui
except ImportError as e:
    raise RuntimeError(RESTART) from e

# 2 · O estoque da aula 10 está ao lado do notebook?
CSV = "cars.csv"
if os.path.exists(CSV):
    print(f"✅ {CSV} já está aqui")
else:
    url = "https://raw.githubusercontent.com/ivanovitchm/aiengineering/main/lessons/week07/cars.csv"
    r = requests.get(url)
    r.raise_for_status()
    open(CSV, "wb").write(r.content)
    print(f"✅ {CSV} baixado")

def server_is_up() -> bool:
    try:
        return requests.get(f"{OLLAMA_URL}/api/version", timeout=2).ok
    except requests.RequestException:
        return False

# 3 · O programa ollama existe nesta máquina?
if shutil.which("ollama"):
    print("✅ Ollama já instalado")
elif IN_COLAB:
    print("⏳ instalando o Ollama (cerca de 1 minuto)...")
    sh("apt-get install -y -qq zstd > /dev/null 2>&1 || "
       "(apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq zstd > /dev/null 2>&1)")
    sh("curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 2")
    if not shutil.which("ollama"):
        raise RuntimeError("a instalação falhou. Rode a célula de novo.")
    print("✅ Ollama instalado")
else:
    raise RuntimeError("Ollama não encontrado. Instale em https://ollama.com/download e rode esta célula de novo.")

# 4 · O servidor do Ollama está respondendo?
if server_is_up():
    print("✅ servidor do Ollama já está de pé")
else:
    print("⏳ subindo o servidor do Ollama...")
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT,
                     start_new_session=True)  # continua vivo depois que a célula termina
    for _ in range(30):
        if server_is_up():
            break
        time.sleep(1)
    if not server_is_up():
        raise RuntimeError("o servidor não respondeu. Veja o arquivo ollama.log")
    print("✅ servidor do Ollama de pé")

# 5 · O modelo já foi baixado?
installed = [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags").json()["models"]]
if MODEL in installed:
    print(f"✅ {MODEL} já baixado")
else:
    print(f"⏳ baixando {MODEL} (~2 GB). Se a conexão cair, rode a célula de novo: o download continua.")
    sh(f"ollama pull {MODEL}")
    print(f"✅ {MODEL} baixado")

version = requests.get(f"{OLLAMA_URL}/api/version").json()["version"]
print(f"\n🚀 pronto · FastMCP {fastmcp.__version__} · Ollama {version} · modelo {MODEL} · "
      f"{'Colab' if IN_COLAB else 'máquina local'}")

✅ FastMCP 4.1.4 já instalado
✅ cars.csv já está aqui
✅ Ollama já instalado
✅ servidor do Ollama já está de pé
✅ qwen2.5:3b já baixado

🚀 pronto · FastMCP 4.1.4 · Ollama 0.40.11 · modelo qwen2.5:3b · máquina local


In [2]:
# @title 🔧 Utilitários: os mesmos da aula 10 (a chamada ao modelo e duas funções para imprimir)
import json

def llm(messages, tools=None):
    """Uma chamada ao /api/chat do Ollama. Devolve o JSON inteiro da resposta."""
    body = {"model": MODEL, "messages": messages, "stream": False,
            "options": {"temperature": 0,          # zero: a mesma entrada dá (quase) a mesma saída
                        "num_predict": 600}}       # teto de tokens por resposta, para nada travar
    if tools:
        body["tools"] = tools
    r = requests.post(f"{OLLAMA_URL}/api/chat", json=body, timeout=300)
    r.raise_for_status()
    return r.json()

def show(obj, limit=1600):
    """Imprime um dicionário como JSON legível, cortado se for longo."""
    text = json.dumps(obj, ensure_ascii=False, indent=2)
    print(text if len(text) <= limit else text[:limit] + "\n  ... (cortado)")

ICON = {"system": "⚙️ ", "user": "👤", "assistant": "🤖", "tool": "🔧"}

def show_messages(messages):
    """A conversa inteira, uma linha por mensagem."""
    for i, m in enumerate(messages):
        if m.get("tool_calls"):
            what = "pede " + " e ".join(f'{c["function"]["name"]}({json.dumps(c["function"]["arguments"], ensure_ascii=False)})'
                                        for c in m["tool_calls"])
        else:
            what = m["content"].replace("\n", " ")
        print(f"[{i}] {ICON[m['role']]} {m['role']:<9} │ {what[:120]}{'…' if len(what) > 120 else ''}")

print("utilitários prontos")

utilitários prontos


---
# Parte 1 · O servidor

Na aula 10, cada ferramenta tinha duas partes: a função Python e uma descrição em JSON Schema escrita à mão. Com o FastMCP, a descrição sai da própria função:

| Na descrição da ferramenta | Vem de |
|---|---|
| **nome** | o nome da função |
| **descrição** | a docstring |
| **parâmetros** | as anotações de tipo (`float`, `str`, `list[str]`...) |

A célula abaixo grava o arquivo `server.py`. O corpo das três funções é o mesmo da aula 10. As linhas novas estão marcadas com `# ←`. Além das ferramentas, o arquivo tem as outras duas primitivas do slide, um **resource** e um **prompt**.

In [3]:
%%writefile server.py
from typing import Literal
import pandas as pd
from fastmcp import FastMCP                                   # ←

cars = pd.read_csv("cars.csv")
mcp = FastMCP("estoque-natal")                                # ← o servidor, com um nome

# ─── TOOLS: o modelo decide quando chamar ───

@mcp.tool                                                     # ← isto substitui o JSON Schema escrito à mão
def search_cars(max_price: float, body: Literal["SUV", "sedan", "hatch"] | None = None) -> dict:
    """Search the used-car stock by budget and body type.
    Returns how many cars matched and the 5 newest ones (car_id, model, year, km, price)."""
    r = cars[cars.price_brl <= max_price]
    if body is not None:
        r = r[r.body == body]
    r = r.sort_values(["year", "km"], ascending=[False, True])
    cols = ["car_id", "brand", "model", "year", "km", "price_brl", "transmission"]
    return {"total_found": len(r), "cars": r[cols].head(5).to_dict("records")}

@mcp.tool                                                     # ←
def car_details(car_id: str) -> dict:
    """Get all the details of one car: trunk size (liters), fuel consumption (km/l),
    color and city. Use it with a car_id returned by search_cars."""
    r = cars[cars.car_id == car_id]
    if r.empty:
        return {"error": f"no car with id {car_id}"}
    return r.to_dict("records")[0]

@mcp.tool                                                     # ←
def compare_cars(car_ids: list[str]) -> dict:
    """Compare 2 to 4 cars side by side (year, km, price, transmission, trunk size, consumption).
    Use it when the user wants to choose between specific cars."""
    r = cars[cars.car_id.isin(car_ids)]
    if r.empty:
        return {"error": f"none of these ids exist: {car_ids}"}
    cols = ["car_id", "brand", "model", "year", "km", "price_brl", "transmission", "trunk_l", "consumption_kml"]
    def winners(col, highest):
        target = r[col].max() if highest else r[col].min()
        return list(r[r[col] == target].car_id)              # mais de um id = empate
    best = {"largest_trunk": winners("trunk_l", True),
            "most_economical": winners("consumption_kml", True),   # mais km por litro = consome menos
            "cheapest": winners("price_brl", False),
            "newest": winners("year", True),
            "lowest_km": winners("km", False)}
    return {"cars": r[cols].to_dict("records"), "best": best,
            "missing": [i for i in car_ids if i not in set(r.car_id)]}


# ─── RESOURCE: a aplicação lê, por um endereço ───

@mcp.resource("estoque://schema")                             # ←
def schema() -> str:
    """The columns of the stock table."""
    return ", ".join(cars.columns)


# ─── PROMPT: o usuário escolhe usar ───

@mcp.prompt                                                   # ←
def recomendar(orcamento: int, carroceria: str) -> str:
    """Ask for a recommendation with budget and body type filled in."""
    return f"Quero um {carroceria} até R$ {orcamento}. Busque no estoque e recomende 3 carros, citando o car_id."


if __name__ == "__main__":
    mcp.run()                                                 # ← fala MCP pelo stdio (entrada e saída padrão)

Overwriting server.py


### 🔍 O que aconteceu

- **A célula só gravou um arquivo.** Nada está rodando. Quem vai subir o servidor é o cliente, que o inicia como subprocesso e conversa com ele pela entrada e pela saída padrão (o transporte **stdio** do slide).
- **Sumiram a lista `TOOLS` e o dicionário `FUNCTIONS`** da aula 10, umas 40 linhas de JSON escritas à mão. O decorador `@mcp.tool` gera as duas coisas.
- **O `enum` agora é um tipo.** Na aula 10 escrevemos `"enum": ["SUV", "sedan", "hatch"]`. Aqui, `Literal["SUV", "sedan", "hatch"]` diz o mesmo para o Python e para o schema.

---
# Parte 2 · Inspecionar, sem LLM nenhum

Na sua máquina, o **MCP Inspector** do slide abre uma interface no navegador:

```bash
npx @modelcontextprotocol/inspector python server.py
```

No Colab não há navegador para ele, então usamos o equivalente de linha de comando que vem com o FastMCP, `fastmcp list` e `fastmcp call`. Cada comando sobe o `server.py`, faz uma pergunta e desliga o servidor. O `2>/dev/null` no fim só esconde o log que o servidor escreve ao subir. Se um comando não mostrar nada, rode-o sem esse final para ver o erro.

🔮 **Antes de rodar.** Nunca escrevemos um JSON Schema. O que vai aparecer como entrada da `search_cars`?

In [4]:
!fastmcp list server.py --resources --prompts --input-schema 2>/dev/null

Tools (3)

  search_cars(max_price: float, body: str | None = ...) -> dict
    Search the used-car stock by budget and body type.
Returns how many cars matched and the 5 newest ones (car_id, model, year, km, 
price).
    Input: {"type": "object", "additionalProperties": false, "properties": 
{"max_price": {"type": "number"}, "body": {"anyOf": [{"enum": ["SUV", "sedan", 
"hatch"], "type": "string"}, {"type": "null"}], "default": null}}, "required": 
["max_price"]}

  car_details(car_id: str) -> dict
    Get all the details of one car: trunk size (liters), fuel consumption 
(km/l),
color and city. Use it with a car_id returned by search_cars.
    Input: {"type": "object", "additionalProperties": false, "properties": 
{"car_id": {"type": "string"}}, "required": ["car_id"]}

  compare_cars(car_ids: list) -> dict
    Compare 2 to 4 cars side by side (year, km, price, transmission, trunk size,
consumption).
Use it when the user wants to choose between specific cars.
    Input: {"type": "obje

### 🔍 O que aconteceu

O servidor se descreveu sozinho: três ferramentas, um resource e um prompt. Em `Input` está o JSON Schema que o FastMCP gerou a partir das anotações de tipo. O `enum` do `Literal` está lá. Como `body` é opcional (`| None = None`), o schema diz "ou um desses três textos, ou `null`" (`anyOf`). A descrição de cada ferramenta é a docstring.

Agora, chamar uma ferramenta, ler o resource e pegar o prompt, como faríamos nas abas do Inspector:

In [5]:
print("🔧 TOOL search_cars")
!fastmcp call server.py search_cars max_price=80000 body=SUV 2>/dev/null
print("\n📄 RESOURCE estoque://schema")
!fastmcp call server.py estoque://schema 2>/dev/null
print("\n💬 PROMPT recomendar")
!fastmcp call server.py --prompt recomendar orcamento=80000 carroceria=SUV 2>/dev/null

🔧 TOOL search_cars
{
  "total_found": 22,
  "cars": [
    {
      "car_id": "c082",
      "brand": "Renault",
      "model": "Duster",
      "year": 2021,
      "km": 56500,
      "price_brl": 75000,
      "transmission": "automatic"
    },
    {
      "car_id": "c081",
      "brand": "Renault",
      "model": "Duster",
      "year": 2021,
      "km": 67700,
      "price_brl": 71000,
      "transmission": "manual"
    },
    {
      "car_id": "c109",
      "brand": "Volkswagen",
      "model": "Nivus",
      "year": 2021,
      "km": 70300,
      "price_brl": 75500,
      "transmission": "manual"
    },
    {
      "car_id": "c061",
      "brand": "Jeep",
      "model": "Renegade",
      "year": 2020,
      "km": 82700,
      "price_brl": 76500,
      "transmission": "manual"
    },
    {
      "car_id": "c003",
      "brand": "Chevrolet",
      "model": "Tracker",
      "year": 2019,
      "km": 91000,
      "price_brl": 74500,
      "transmission": "automatic"
    }
  ]
}

📄 RESOURCE

### 🔍 O que aconteceu

A busca devolveu 22 SUVs, com **`c082`, `c081` e `c109`** no topo, o mesmo gabarito da aula 10. E não houve modelo nenhum na conversa. É a regra da aula 10, *"uma ferramenta se testa sem LLM, como qualquer função"*, agora para um servidor inteiro. É assim que a **fase 2 do Trabalho 2** é validada.

Repare também na diferença entre as três primitivas. A ferramenta **executa** uma busca. O resource **devolve um conteúdo** pelo endereço `estoque://schema`. E o prompt **devolve uma mensagem pronta**, com os campos preenchidos, para o usuário mandar ao modelo.

---
# Parte 3 · JSON-RPC na mão

O `fastmcp call` escondeu as mensagens. Agora **nós** fazemos o papel do cliente. Subimos o `server.py` como subprocesso e escrevemos cada mensagem JSON-RPC na entrada dele, uma por linha. A resposta vem na saída, também uma por linha. Sem biblioteca nenhuma, só `subprocess` e `json`.

Na especificação de 2026, não há aperto de mão inicial. **Cada pedido leva no `_meta` a versão do protocolo, as capacidades do cliente e quem ele é.** O servidor não guarda nada entre um pedido e outro.

In [6]:
server = subprocess.Popen([sys.executable, "server.py"],          # o cliente sobe o servidor
                          stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                          stderr=subprocess.DEVNULL, text=True)

META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "aula11", "version": "1.0"}}
next_id = 1

def rpc(method, params=None, meta=True, limit=1600):
    """Envia um pedido JSON-RPC ao servidor, numa linha, e lê a resposta, na linha seguinte."""
    global next_id
    msg = {"jsonrpc": "2.0", "id": next_id, "method": method, "params": dict(params or {})}
    if meta:
        msg["params"]["_meta"] = META
    next_id += 1
    print("📤 ENVIAMOS")
    show({**msg, "params": {**msg["params"], "_meta": "{...}"}} if meta else msg)   # o _meta vai inteiro; só não o imprimimos
    server.stdin.write(json.dumps(msg) + "\n")
    server.stdin.flush()
    resp = json.loads(server.stdout.readline())
    print("\n📥 RECEBEMOS")
    show(resp, limit)
    return resp

resp = rpc("server/discover")

📤 ENVIAMOS
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "server/discover",
  "params": {
    "_meta": "{...}"
  }
}

📥 RECEBEMOS
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "_meta": {
      "io.modelcontextprotocol/serverInfo": {
        "name": "estoque-natal",
        "version": "4.1.4"
      }
    },
    "ttlMs": 0,
    "cacheScope": "private",
    "supportedVersions": [
      "2026-07-28"
    ],
    "capabilities": {
      "logging": {},
      "prompts": {
        "listChanged": false
      },
      "resources": {
        "subscribe": false,
        "listChanged": false
      },
      "tools": {
        "listChanged": false
      },
      "extensions": {
        "io.modelcontextprotocol/ui": {}
      }
    },
    "resultType": "complete"
  }
}


### 🔍 O que aconteceu

Para não poluir a tela, o `_meta` aparece como `{...}` no que enviamos, mas ele vai inteiro em todo pedido. A resposta tem o **mesmo `id`** do pedido. É assim que o cliente sabe a qual pedido ela responde. Em `capabilities`, o servidor diz o que oferece (`tools`, `resources` e `prompts`). Em `supportedVersions`, diz qual versão do protocolo fala.

O `server/discover` é opcional. Um cliente pode ir direto ao que interessa: listar as ferramentas.

In [7]:
resp = rpc("tools/list", limit=600)

print("\n🔎 SÓ A ENTRADA DA search_cars")
show(resp["result"]["tools"][0]["inputSchema"])

📤 ENVIAMOS
{
  "jsonrpc": "2.0",
  "id": 2,
  "method": "tools/list",
  "params": {
    "_meta": "{...}"
  }
}

📥 RECEBEMOS
{
  "jsonrpc": "2.0",
  "id": 2,
  "result": {
    "cacheScope": "private",
    "resultType": "complete",
    "tools": [
      {
        "_meta": {
          "fastmcp": {
            "tags": []
          }
        },
        "description": "Search the used-car stock by budget and body type.\nReturns how many cars matched and the 5 newest ones (car_id, model, year, km, price).",
        "inputSchema": {
          "type": "object",
          "additionalProperties": false,
          "properties": {
            "max_price": {
              "type": "number"
            },
            "body": {
    
  ... (cortado)

🔎 SÓ A ENTRADA DA search_cars
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "max_price": {
      "type": "number"
    },
    "body": {
      "anyOf": [
        {
          "enum": [
            "SUV",
            "sedan",
     

### 🔍 O que aconteceu

É o mesmo schema que o `fastmcp list` mostrou, agora como ele viaja: um campo `inputSchema` dentro de cada ferramenta da lista. **Esse JSON é o que vai para o modelo.** Na Parte 5, é ele que entra no lugar da lista `TOOLS` da aula 10.

Agora, a chamada.

In [8]:
resp = rpc("tools/call", {"name": "search_cars", "arguments": {"max_price": 80000, "body": "SUV"}})

📤 ENVIAMOS
{
  "jsonrpc": "2.0",
  "id": 3,
  "method": "tools/call",
  "params": {
    "name": "search_cars",
    "arguments": {
      "max_price": 80000,
      "body": "SUV"
    },
    "_meta": "{...}"
  }
}

📥 RECEBEMOS
{
  "jsonrpc": "2.0",
  "id": 3,
  "result": {
    "content": [
      {
        "text": "{\"total_found\":22,\"cars\":[{\"car_id\":\"c082\",\"brand\":\"Renault\",\"model\":\"Duster\",\"year\":2021,\"km\":56500,\"price_brl\":75000,\"transmission\":\"automatic\"},{\"car_id\":\"c081\",\"brand\":\"Renault\",\"model\":\"Duster\",\"year\":2021,\"km\":67700,\"price_brl\":71000,\"transmission\":\"manual\"},{\"car_id\":\"c109\",\"brand\":\"Volkswagen\",\"model\":\"Nivus\",\"year\":2021,\"km\":70300,\"price_brl\":75500,\"transmission\":\"manual\"},{\"car_id\":\"c061\",\"brand\":\"Jeep\",\"model\":\"Renegade\",\"year\":2020,\"km\":82700,\"price_brl\":76500,\"transmission\":\"manual\"},{\"car_id\":\"c003\",\"brand\":\"Chevrolet\",\"model\":\"Tracker\",\"year\":2019,\"km\":91000,

### 🔍 O que aconteceu

O resultado vem duas vezes. Em `content`, como **texto**, que é o que vai para o modelo ler. Em `structuredContent`, como **objeto**, para um programa usar sem precisar fazer o parse do texto. E `isError: false` diz que a ferramenta rodou sem erro.

## Dois tipos de erro

🔮 **Antes de rodar.** Vamos mandar dois pedidos errados: uma chamada com `max_price: "barato"` e um método que não existe, `tools/run`. As duas respostas vão ter a mesma forma?

In [9]:
print("═══ 1 · argumento errado ═══\n")
rpc("tools/call", {"name": "search_cars", "arguments": {"max_price": "barato"}})

print("\n═══ 2 · método que não existe ═══\n")
resp = rpc("tools/run")

═══ 1 · argumento errado ═══

📤 ENVIAMOS
{
  "jsonrpc": "2.0",
  "id": 4,
  "method": "tools/call",
  "params": {
    "name": "search_cars",
    "arguments": {
      "max_price": "barato"
    },
    "_meta": "{...}"
  }
}

📥 RECEBEMOS
{
  "jsonrpc": "2.0",
  "id": 4,
  "result": {
    "content": [
      {
        "text": "1 validation error for call[search_cars]\nmax_price\n  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='barato', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.14/v/float_parsing",
        "type": "text"
      }
    ],
    "isError": true,
    "resultType": "complete",
    "_meta": {
      "io.modelcontextprotocol/serverInfo": {
        "name": "estoque-natal",
        "version": "4.1.4"
      }
    }
  }
}

═══ 2 · método que não existe ═══

📤 ENVIAMOS
{
  "jsonrpc": "2.0",
  "id": 5,
  "method": "tools/run",
  "params": {
    "_meta": "{...}"
  }
}

📥 RECEBEMOS
{
  "jsonrpc": "2.0"

### 🔍 O que aconteceu

As duas formas são diferentes, como no slide dos dois tipos de erro.

| | Onde aparece | Quem lê | Exemplo típico |
|---|---|---|---|
| **1 · Erro de execução** | dentro de `result`, com `isError: true` | **o modelo**, que pode ler a mensagem e tentar de novo com outro argumento | argumento inválido, carro não encontrado |
| **2 · Erro de protocolo** | no campo `error` do JSON-RPC, com um código (`-32601`, *Method not found*) | **o programa** do cliente, porque é bug de código | método inexistente, mensagem malformada |

No primeiro, a mensagem do Pydantic explica o problema (*"Input should be a valid number"*). Um modelo que lê isso consegue se corrigir.

## Cada pedido se sustenta sozinho

O que acontece se mandarmos um pedido **sem** o `_meta`? E se tentarmos o aperto de mão das versões antigas, o `initialize`?

In [10]:
print("═══ 3 · sem o _meta ═══\n")
rpc("tools/list", meta=False)

print("\n═══ 4 · o initialize de 2025 ═══\n")
rpc("initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                   "clientInfo": {"name": "aula11", "version": "1.0"}}, meta=False)

server.stdin.close()          # fecha a entrada: o servidor entende que acabou e termina
server.wait(timeout=10)
print("\n🛑 servidor encerrado")

═══ 3 · sem o _meta ═══

📤 ENVIAMOS
{
  "jsonrpc": "2.0",
  "id": 6,
  "method": "tools/list",
  "params": {}
}

📥 RECEBEMOS
{
  "jsonrpc": "2.0",
  "id": 6,
  "error": {
    "code": -32602,
    "message": "params._meta must be an object carrying the required 'io.modelcontextprotocol/protocolVersion' and 'io.modelcontextprotocol/clientCapabilities' envelope keys"
  }
}

═══ 4 · o initialize de 2025 ═══

📤 ENVIAMOS
{
  "jsonrpc": "2.0",
  "id": 7,
  "method": "initialize",
  "params": {
    "protocolVersion": "2025-06-18",
    "capabilities": {},
    "clientInfo": {
      "name": "aula11",
      "version": "1.0"
    }
  }
}

📥 RECEBEMOS
{
  "jsonrpc": "2.0",
  "id": 7,
  "error": {
    "code": -32022,
    "message": "connection is serving the 2026-07-28 protocol; the initialize handshake is not accepted",
    "data": {
      "supported": [
        "2026-07-28"
      ],
      "requested": "2025-06-18"
    }
  }
}

🛑 servidor encerrado


### 🔍 O que aconteceu

Os dois foram recusados com um erro de protocolo.

- **Sem `_meta`**, o servidor não sabe qual versão o cliente fala, então não responde. Como não existe sessão, ele não tem onde ter guardado isso.
- **O `initialize`** era o aperto de mão obrigatório até 2025. Este servidor fala só a versão `2026-07-28` e responde que esse aperto de mão não é aceito. Se você encontrar um tutorial antigo começando por `initialize`, é por isso.

Foram sete pedidos, e todos tinham a mesma estrutura: `jsonrpc`, `id`, `method` e `params` na ida, e `result` ou `error` na volta. **Isso é o protocolo inteiro.** O resto é o significado de cada método.

---
# Parte 4 · Um cliente MCP em Python

Escrever JSON-RPC na mão serve para entender o protocolo, não para trabalhar com ele. O `Client` do FastMCP faz tudo o que a Parte 3 fez: sobe o servidor, monta o `_meta`, numera os `id` e transforma as respostas em objetos Python.

O cliente é **assíncrono**, por isso o `async with` e os `await`. No notebook, eles funcionam direto na célula.

In [11]:
from pathlib import Path
from fastmcp import Client

async with Client(Path("server.py")) as client:              # sobe o server.py como subprocesso (stdio)
    tools = await client.list_tools()                         # tools/list
    print("🔧 ferramentas:", [t.name for t in tools])

    res = await client.call_tool("search_cars", {"max_price": 80_000, "body": "SUV"})   # tools/call
    print("\n🔧 call_tool('search_cars') →  res.data, já como dicionário Python")
    show(res.data, limit=500)

    content = await client.read_resource("estoque://schema")  # resources/read
    print("\n📄 read_resource('estoque://schema') →", content[0].text)

    prompt = await client.get_prompt("recomendar", {"orcamento": 80000, "carroceria": "SUV"})   # prompts/get
    print("\n💬 get_prompt('recomendar') →", prompt.messages[0].content.text)

print("\n🛑 fora do async with, o servidor já foi encerrado")

🔧 ferramentas: ['search_cars', 'car_details', 'compare_cars']

🔧 call_tool('search_cars') →  res.data, já como dicionário Python
{
  "total_found": 22,
  "cars": [
    {
      "car_id": "c082",
      "brand": "Renault",
      "model": "Duster",
      "year": 2021,
      "km": 56500,
      "price_brl": 75000,
      "transmission": "automatic"
    },
    {
      "car_id": "c081",
      "brand": "Renault",
      "model": "Duster",
      "year": 2021,
      "km": 67700,
      "price_brl": 71000,
      "transmission": "manual"
    },
    {
      "car_id": "c109",
      "brand": "Volkswagen",
      "model": "Nivus",
      "year
  ... (cortado)

📄 read_resource('estoque://schema') → car_id, brand, model, body, year, km, price_brl, transmission, fuel, consumption_kml, trunk_l, color, city

💬 get_prompt('recomendar') → Quero um SUV até R$ 80000. Busque no estoque e recomende 3 carros, citando o car_id.

🛑 fora do async with, o servidor já foi encerrado


### 🔍 O que aconteceu

Cada `await` virou um pedido JSON-RPC igual aos da Parte 3 (o comentário de cada linha diz qual). Com isso, o servidor ficou ligado só enquanto o bloco `async with` rodou.

As três primitivas, com os três tipos de controle do slide:

| Primitiva | Método do cliente | Quem decide usar |
|---|---|---|
| **Tool** | `call_tool` | o **modelo** |
| **Resource** | `read_resource` | a **aplicação** |
| **Prompt** | `get_prompt` | o **usuário** |

---
# Parte 5 · O agente da aula 10, com ferramentas MCP

O loop da aula 10 muda em **dois** lugares, e só neles:

| | Aula 10 | Aula 11 |
|---|---|---|
| **As descrições das ferramentas** | a lista `TOOLS`, escrita à mão | `client.list_tools()`, perguntando ao servidor |
| **A execução** | `FUNCTIONS[name](**args)`, uma função no notebook | `client.call_tool(name, args)`, um pedido ao servidor |

O *system prompt*, o pedido, o `max_steps` e o resto do loop continuam iguais.

## 5.1 · As descrições vêm do servidor

O Ollama espera cada ferramenta no formato que usamos na aula 10. A ferramenta MCP já tem as mesmas três informações. Só muda o nome das chaves.

In [12]:
def to_ollama(tool):
    """Uma ferramenta MCP no formato de ferramenta do Ollama."""
    return {"type": "function",
            "function": {"name": tool.name,
                         "description": tool.description,
                         "parameters": tool.input_schema}}     # no protocolo, o campo se chama inputSchema

async with Client(Path("server.py")) as client:
    TOOLS = [to_ollama(t) for t in await client.list_tools()]

print(f"{len(TOOLS)} ferramentas vindas do servidor. A primeira:\n")
show(TOOLS[0])

3 ferramentas vindas do servidor. A primeira:

{
  "type": "function",
  "function": {
    "name": "search_cars",
    "description": "Search the used-car stock by budget and body type.\nReturns how many cars matched and the 5 newest ones (car_id, model, year, km, price).",
    "parameters": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "max_price": {
          "type": "number"
        },
        "body": {
          "anyOf": [
            {
              "enum": [
                "SUV",
                "sedan",
                "hatch"
              ],
              "type": "string"
            },
            {
              "type": "null"
            }
          ],
          "default": null
        }
      },
      "required": [
        "max_price"
      ]
    }
  }
}


Compare com a seção 3.2 da aula 10: a estrutura é a mesma (`name`, `description` e `parameters`), mas desta vez ninguém escreveu esse JSON.

## 5.2 · O loop, agora falando com o servidor

O `run_agent` abaixo é o da aula 10. As duas linhas que mudaram estão marcadas com `# ←`.

In [13]:
SYSTEM = """You are a used-car buying assistant in Natal, Brazil. Answer in the user's language.
Use the tools to look up real listings; never invent a car.
Always cite the car_id of every car you mention."""

PEDIDO = "Quero um SUV para a família, até uns 80 mil. O que você recomenda?"

def summary(result):
    """Uma linha sobre o resultado de uma ferramenta, para o trace."""
    if "error" in result:
        return "erro: " + result["error"]
    if "total_found" in result:                                        # search_cars
        return f'{result["total_found"]} encontrados; devolvemos {[c["car_id"] for c in result["cars"]]}'
    if "best" in result:                                               # compare_cars
        return "melhor em cada critério: " + ", ".join(f"{k} {'/'.join(v)}" for k, v in result["best"].items())
    return (f'{result["brand"]} {result["model"]} {result["year"]}: porta-malas {result["trunk_l"]} L, '   # car_details
            f'consumo {result["consumption_kml"]} km/l, {result["color"]}, {result["city"]}')

async def run_agent(pedido, max_steps=5):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": pedido}]
    print(f"👤 {pedido}\n")
    async with Client(Path("server.py")) as client:                   # ← o agente agora é um host MCP
        tools = [to_ollama(t) for t in await client.list_tools()]     # ← as descrições vêm do servidor
        for volta in range(1, max_steps + 1):
            print(f"── volta {volta} " + "─" * 60)
            msg = llm(messages, tools=tools)["message"]                # REASON: o modelo decide
            messages.append(msg)
            if not msg.get("tool_calls"):                              # texto em vez de ferramenta: fim do loop
                print("🤖 o modelo respondeu com texto → fim do loop\n")
                print(msg["content"])
                return messages
            for c in msg["tool_calls"]:
                name, args = c["function"]["name"], c["function"]["arguments"]
                print(f"🤖 o modelo pediu   → {name}({json.dumps(args, ensure_ascii=False)})")
                res = await client.call_tool(name, args, raise_on_error=False)   # ← ACT: o servidor executa
                if res.is_error:
                    print(f"📡 o servidor       → isError: {res.content[0].text[:80]}")
                else:
                    print(f"📡 o servidor       → {summary(res.data)}")
                messages.append({"role": "tool", "tool_name": name,             # OBSERVE: o resultado volta
                                 "content": res.content[0].text})
    print(f"(parei depois de {max_steps} voltas sem uma resposta final)")
    return messages

conversa = await run_agent(PEDIDO)

👤 Quero um SUV para a família, até uns 80 mil. O que você recomenda?

── volta 1 ────────────────────────────────────────────────────────────
🤖 o modelo pediu   → search_cars({"max_price": 80000, "body": "SUV"})
📡 o servidor       → 22 encontrados; devolvemos ['c082', 'c081', 'c109', 'c061', 'c003']
── volta 2 ────────────────────────────────────────────────────────────
🤖 o modelo respondeu com texto → fim do loop

Com base nos resultados da pesquisa, aqui estão algumas opções de SUV que atendem aos seus critérios:

1. Renault Duster (car_id: c082) - Ano: 2021, KM: 56,500, Preço: R$ 75.000
2. Renault Duster (car_id: c081) - Ano: 2021, KM: 67,700, Preço: R$ 71.000
3. Volkswagen Nivus (car_id: c109) - Ano: 2021, KM: 70,300, Preço: R$ 75.500
4. Jeep Renegade (car_id: c061) - Ano: 2020, KM: 82,700, Preço: R$ 76.500
5. Chevrolet Tracker (car_id: c003) - Ano: 2019, KM: 91,000, Preço: R$ 74.500

Estes são os 5 SUVs mais recentes e dentro do seu orçamento. Você pode optar por um comparando-os 

### 🔍 O que aconteceu

O mesmo trace da aula 10. Na volta 1, o modelo pediu `search_cars` com `max_price: 80000` e `body: "SUV"`. Na volta 2, respondeu com texto, e a lista começa por **`c082`, `c081` e `c109`**, o gabarito.

A diferença está na linha 📡. Quem executou a busca foi o **servidor**, outro processo, que recebeu um `tools/call` igual ao da Parte 3. O loop não sabe se a ferramenta é uma função local ou um servidor do outro lado do mundo, e o modelo também não. Ele continua lendo nome, descrição e parâmetros, e escrevendo um pedido em JSON.

Repare num detalhe: `raise_on_error=False`. Se a ferramenta falhar, o erro de execução (`isError: true`) volta para o modelo como resultado, igual ao `try/except` da aula 10. Um erro vira observação, não derruba o loop.

## 5.3 · As outras ferramentas, pelo mesmo servidor

Uma pergunta que precisa da `car_details` e outra que precisa de duas voltas, com a busca e depois a comparação.

In [14]:
conversa = await run_agent("Em que cidade está o c109 e qual é a cor dele?")
print("\n" + "═" * 72 + "\n")
conversa = await run_agent("Quero um SUV até 80 mil para a família. Compare os dois mais novos e me diga qual você escolheria.")

print("\n📜 A CONVERSA COMPLETA")
show_messages(conversa)

👤 Em que cidade está o c109 e qual é a cor dele?

── volta 1 ────────────────────────────────────────────────────────────
🤖 o modelo pediu   → car_details({"car_id": "c109"})
📡 o servidor       → Volkswagen Nivus 2021: porta-malas 415 L, consumo 12.5 km/l, vermelho, Natal
── volta 2 ────────────────────────────────────────────────────────────
🤖 o modelo respondeu com texto → fim do loop

O carro com o car_id c109 é um Volkswagen Nivus, um SUV da marca Volkswagen. Ele está na cidade de Natal. A cor deste carro é vermelho.

════════════════════════════════════════════════════════════════════════

👤 Quero um SUV até 80 mil para a família. Compare os dois mais novos e me diga qual você escolheria.

── volta 1 ────────────────────────────────────────────────────────────
🤖 o modelo pediu   → search_cars({"max_price": 80000, "body": "SUV"})
📡 o servidor       → 22 encontrados; devolvemos ['c082', 'c081', 'c109', 'c061', 'c003']
── volta 2 ──────────────────────────────────────────────────────

### 🔍 O que aconteceu

Na primeira pergunta, o modelo escolheu `car_details("c109")` lendo só as descrições que vieram do servidor e respondeu que o Nivus está em Natal e é vermelho.

Na segunda, foram três voltas. Primeiro `search_cars`, depois `compare_cars(["c081", "c082"])` com os ids que o próprio modelo tirou da busca, e então a resposta. É o mesmo ReAct da aula 10, e cada ação passou pelo servidor.

Os números continuam vindo das ferramentas e o texto continua sendo do modelo. Durante a preparação, ele escolheu o `c081` dizendo que ele tem "um menor número de quilômetros rodados" e "é o mais novo". O trace mostra outra coisa: em `lowest_km`, quem venceu foi o `c082`, e em `newest` deu empate (`c081/c082`). A escolha em si se defende, porque o `c081` venceu sem empate em `cheapest` e `most_economical`. A justificativa, não. Trocar o transporte das ferramentas não muda essa lição. **O texto se confere.**

---
# Resumo

```
👤 pedido
   │
   ▼
🤖 agente (host + cliente MCP) ── tools/call · JSON-RPC · stdio ──►  📡 server.py (FastMCP)
   ▲                                                                       │
   └─────────────── resultado: content, structuredContent, isError ◄──────┘
   │
   ▼  (quando o modelo responde com texto)
💬 resposta
```

1. **Um servidor MCP é um programa separado** que oferece ferramentas, resources e prompts. Com o FastMCP, são funções Python com um decorador, e a descrição sai da docstring e das anotações de tipo.
2. **O protocolo são mensagens JSON-RPC.** `jsonrpc`, `id`, `method` e `params` na ida, `result` ou `error` na volta. Na versão de 2026, cada pedido leva o próprio `_meta` e não há sessão.
3. **Há dois tipos de erro.** O erro de execução (`isError`) é para o modelo ler. O erro de protocolo (`error`) é para o programa.
4. **O agente quase não muda.** As descrições vêm de `list_tools` e a execução vai por `call_tool`. O loop, o *system prompt* e a resposta são os mesmos da aula 10.

### Daqui para a fase 2 do Trabalho 2

Pegue as três ferramentas da fase 1 do seu grupo (`search`, `details` e `compare`) e transforme-as num servidor MCP, como a Parte 1. Valide o servidor no **MCP Inspector** na sua máquina, listando e chamando as três ferramentas sem LLM, como a Parte 2. Depois troque, no seu agente, a lista de ferramentas e a execução pelas chamadas ao servidor, como a Parte 5. **Prazo: 09/11, no início da aula.**

| Você viu aqui | Volta como |
|---|---|
| O servidor com resources e prompts (Parte 1) | O servidor do seu grupo, fase 2 |
| O mesmo `server.py` em outro host | O Claude Code usando o servidor da fase 2, aulas 14 e 15 |
| O agente como cliente MCP (Parte 5) | O Claude Agent SDK, aula 16 |

Para plugar este servidor no Claude Code, na sua máquina, basta um comando:

```bash
claude mcp add estoque -- python server.py
```